# Guided Challenge - datos y descargas

Este notebook deja en estructuras de Python los datos del Anexo A del `Guided_Challenge_V3.pdf` y agrega ejemplos para descargar o localizar datasets desde las fuentes publicas citadas en el challenge.

In [ ]:
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 30)

## Datos base del Anexo A

In [ ]:
municipalities = [
    {"municipality": "Puebla", "urban_demand_hm3_year": 95, "agricultural_demand_hm3_year": 8},
    {"municipality": "San Andres Cholula", "urban_demand_hm3_year": 14, "agricultural_demand_hm3_year": 5},
    {"municipality": "Atlixco", "urban_demand_hm3_year": 9, "agricultural_demand_hm3_year": 18},
]

water_sources = [
    {"source": "Valle de Puebla Aquifer", "base_availability_hm3_year": 80, "source_type": "Groundwater"},
    {"source": "Atlixco-Izucar Aquifer", "base_availability_hm3_year": 45, "source_type": "Groundwater"},
]

drought_scenarios = [
    {"scenario": "Normal", "availability_factor": 1.00},
    {"scenario": "Moderate drought", "availability_factor": 0.80},
    {"scenario": "Severe drought", "availability_factor": 0.60},
]

source_demand_network = [
    {"source": "Valle de Puebla Aquifer", "municipality": "Puebla", "allocation_cost": 1.0},
    {"source": "Valle de Puebla Aquifer", "municipality": "San Andres Cholula", "allocation_cost": 1.2},
    {"source": "Valle de Puebla Aquifer", "municipality": "Atlixco", "allocation_cost": 2.5},
    {"source": "Atlixco-Izucar Aquifer", "municipality": "Puebla", "allocation_cost": 2.8},
    {"source": "Atlixco-Izucar Aquifer", "municipality": "San Andres Cholula", "allocation_cost": 2.0},
    {"source": "Atlixco-Izucar Aquifer", "municipality": "Atlixco", "allocation_cost": 1.0},
]

priority_weights = {
    "urban_human_consumption": 10,
    "agricultural_demand": 3,
}

crop_water_requirements = [
    {"crop_type": "Maize", "water_requirement_m3_ha_year": 7000},
    {"crop_type": "Alfalfa", "water_requirement_m3_ha_year": 12000},
    {"crop_type": "Beans", "water_requirement_m3_ha_year": 4500},
    {"crop_type": "Wheat", "water_requirement_m3_ha_year": 5500},
    {"crop_type": "Barley", "water_requirement_m3_ha_year": 4500},
    {"crop_type": "Sorghum", "water_requirement_m3_ha_year": 6000},
    {"crop_type": "Oats", "water_requirement_m3_ha_year": 5000},
    {"crop_type": "Vegetables (generic)", "water_requirement_m3_ha_year": 8000},
    {"crop_type": "Potato", "water_requirement_m3_ha_year": 6500},
    {"crop_type": "Onion", "water_requirement_m3_ha_year": 7500},
    {"crop_type": "Chili Pepper", "water_requirement_m3_ha_year": 6500},
    {"crop_type": "Fruit Trees (generic)", "water_requirement_m3_ha_year": 9000},
]

In [ ]:
municipalities_df = pd.DataFrame(municipalities)
water_sources_df = pd.DataFrame(water_sources)
drought_scenarios_df = pd.DataFrame(drought_scenarios)
network_df = pd.DataFrame(source_demand_network)
crop_requirements_df = pd.DataFrame(crop_water_requirements)

display(municipalities_df)
display(water_sources_df)
display(drought_scenarios_df)
display(network_df)
display(crop_requirements_df)

In [ ]:
effective_availability_df = (
    water_sources_df.merge(drought_scenarios_df, how="cross")
    .assign(
        effective_availability_hm3_year=lambda df: (
            df["base_availability_hm3_year"] * df["availability_factor"]
        )
    )
    [["scenario", "source", "base_availability_hm3_year", "availability_factor", "effective_availability_hm3_year"]]
)

demand_totals = {
    "total_urban_demand_hm3_year": municipalities_df["urban_demand_hm3_year"].sum(),
    "total_agricultural_demand_hm3_year": municipalities_df["agricultural_demand_hm3_year"].sum(),
}

display(effective_availability_df)
demand_totals

In [ ]:
benchmark_instance = {
    "municipalities": municipalities,
    "water_sources": water_sources,
    "drought_scenarios": drought_scenarios,
    "source_demand_network": source_demand_network,
    "priority_weights": priority_weights,
    "crop_water_requirements": crop_water_requirements,
}

benchmark_instance

## Utilidades para descargar datasets

In [ ]:
optional_data_sources = {
    'inegi_marco_geoestadistico': 'https://www.inegi.org.mx/temas/mg/',
    'inegi_censo_2020': 'https://www.inegi.org.mx/programas/ccpv/2020/',
    'conagua_acuiferos': 'https://sigagis.conagua.gob.mx/gas1/sections/Disponibilidad_Acuiferos.html',
    'siap_cierre_agricola': 'https://nube.agricultura.gob.mx/cierre_agricola/',
    'repda': 'https://app.conagua.gob.mx/consultarepda.aspx',
    'smn_monitor_sequia': 'https://smn.conagua.gob.mx/es/climatologia/monitor-de-sequia/monitor-de-sequia-en-mexico',
}

http = requests.Session()
http.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/125 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-MX,es;q=0.9,en;q=0.8',
})

def get_remote_size_mb(url: str) -> float | None:
    try:
        response = http.head(url, allow_redirects=True, timeout=60)
        response.raise_for_status()
        size = response.headers.get('content-length')
        return round(int(size) / (1024 * 1024), 2) if size else None
    except Exception:
        return None

def download_file(url: str, output_path: Path, chunk_size: int = 1024 * 1024, max_size_mb: float | None = None) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    remote_size_mb = get_remote_size_mb(url)
    if max_size_mb is not None and remote_size_mb is not None and remote_size_mb > max_size_mb:
        raise ValueError(f'Archivo demasiado grande: {remote_size_mb} MB. Sube max_size_mb si quieres descargarlo.')

    try:
        from tqdm import tqdm
    except Exception:
        tqdm = None

    with http.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length') or 0)
        downloaded = 0
        progress = tqdm(
            total=total if total else None,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
            desc=output_path.name,
        ) if tqdm else None

        with output_path.open('wb') as fh:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    fh.write(chunk)
                    downloaded += len(chunk)
                    if progress:
                        progress.update(len(chunk))
                    elif total:
                        print(f'\r{downloaded / 1024 / 1024:.1f} MB / {total / 1024 / 1024:.1f} MB', end='')
                    else:
                        print(f'\r{downloaded / 1024 / 1024:.1f} MB descargados', end='')

        if progress:
            progress.close()
        else:
            print()
    return output_path

def find_dataset_links(page_url: str, extensions=('.zip', '.csv', '.xlsx', '.xls', '.geojson', '.shp', '.pdf')) -> pd.DataFrame:
    from html.parser import HTMLParser

    class LinkParser(HTMLParser):
        def __init__(self):
            super().__init__()
            self.links = []
            self._current_href = None
            self._current_text = []

        def handle_starttag(self, tag, attrs):
            if tag.lower() == 'a':
                attrs = dict(attrs)
                self._current_href = attrs.get('href')
                self._current_text = []

        def handle_data(self, data):
            if self._current_href:
                self._current_text.append(data)

        def handle_endtag(self, tag):
            if tag.lower() == 'a' and self._current_href:
                self.links.append({'href': self._current_href, 'text': ' '.join(self._current_text).strip()})
                self._current_href = None
                self._current_text = []

    response = http.get(page_url, timeout=60)
    response.raise_for_status()
    parser = LinkParser()
    parser.feed(response.text)
    rows = []
    for link in parser.links:
        href = urljoin(page_url, link['href'])
        if any(ext.lower() in href.lower() for ext in extensions):
            rows.append({'text': link['text'], 'url': href})
    return pd.DataFrame(rows).drop_duplicates('url') if rows else pd.DataFrame(columns=['text', 'url'])

def inegi_product_maps(theme_id='213', state_id='', page_size=20, page_number=0) -> pd.DataFrame:
    payload = {
        'enti': state_id, 'muni': '', 'loca': '', 'tema': theme_id, 'titg': '', 'esca': '',
        'form': '', 'edic': '', 'seri': '', 'prog': '', 'clave': '', 'rango': '', 'busc': '',
        'tipoB': 1, 'adv': False, 'wordag': '', 'mkeys': '', 'mageo': '', 'formIncl': '',
        'formExcl': '', 'orden': 4, 'desc': True, 'pag': page_number, 'tam': page_size,
    }
    url = 'https://www.inegi.org.mx/app/api/productos/interna_v2/componente/mapas/lista/resultados/'
    response = http.post(url, json=payload, timeout=60, headers={'Content-Type': 'application/json', 'Referer': optional_data_sources['inegi_marco_geoestadistico']})
    response.raise_for_status()
    data = response.json()
    rows = []
    for item in data.get('list', {}).get('mapas', []):
        for formato in item.get('formatos', []):
            raw_url = (formato.get('url') or {}).get('valor')
            rows.append({
                'title': item.get('titulo'),
                'entity': item.get('entidad'),
                'edition': item.get('edicion'),
                'format': formato.get('extension'),
                'size': formato.get('peso'),
                'url': urljoin('https://www.inegi.org.mx/', raw_url) if raw_url else None,
                'coverage': item.get('cobertura_temp'),
            })
    return pd.DataFrame(rows)

def extract_jsonld_downloads(page_url: str) -> pd.DataFrame:
    import json as _json
    import re as _re

    response = http.get(page_url, timeout=60)
    response.raise_for_status()
    rows = []
    for block in _re.findall(r'<script[^>]+application/ld\+json[^>]*>(.*?)</script>', response.text, flags=_re.I | _re.S):
        data = _json.loads(block.strip())
        distributions = data.get('distribution', [])
        if isinstance(distributions, dict):
            distributions = [distributions]
        for dist in distributions:
            rows.append({
                'name': data.get('name'),
                'format': dist.get('encodingFormat'),
                'url': dist.get('contentUrl'),
            })
    return pd.DataFrame(rows)

optional_data_sources


### INEGI - Marco Geoestadistico

La pagina `https://www.inegi.org.mx/temas/mg/` no publica enlaces directos en el HTML inicial; carga los productos con el API interno de mapas. Esta celda consulta ese API para obtener URLs descargables reales.

In [ ]:
mg_links = inegi_product_maps(theme_id='213', page_size=20)
display(mg_links.head(20))

# El Marco Geoestadistico nacional en SHP puede pesar varios GB.
# Por defecto NO se descarga automaticamente para evitar una celda aparentemente congelada.
mg_shp = mg_links[
    mg_links['title'].str.contains('Marco Geoestadístico|Marco Geoestadistico', case=False, regex=True, na=False)
    & mg_links['format'].str.contains('SHP', case=False, na=False)
].sort_values('edition', ascending=False)

if mg_shp.empty:
    print('No se encontro un SHP de Marco Geoestadistico en la respuesta del API.')
else:
    selected = mg_shp.iloc[0]
    url = selected['url']
    print('Descarga disponible:')
    print('Titulo:', selected['title'])
    print('Edicion:', selected['edition'])
    print('Tamaño INEGI:', selected.get('size'), selected.get('size_unit'))
    print('URL:', url)

    DOWNLOAD_MG = False  # Cambia a True si quieres descargar el ZIP grande.
    if DOWNLOAD_MG:
        output = DATA_DIR / 'inegi_marco_geoestadistico' / Path(url).name
        print(download_file(url, output, max_size_mb=None))


### INEGI - Censo de Poblacion y Vivienda 2020

La pagina del CPV 2020 incluye un bloque JSON-LD con un `contentUrl` directo al ZIP CSV de datos abiertos. Esta celda lo extrae y lo descarga.

In [ ]:
censo_links = extract_jsonld_downloads(optional_data_sources['inegi_censo_2020'])
display(censo_links)

if censo_links.empty:
    print('No se encontro contentUrl de datos abiertos en el JSON-LD de la pagina.')
else:
    url = censo_links.iloc[0]['url']
    print('Descarga disponible:', url)
    print('Tamaño remoto aproximado:', get_remote_size_mb(url), 'MB')

    DOWNLOAD_CENSO = False  # Cambia a True para descargar el ZIP CSV.
    if DOWNLOAD_CENSO:
        output = DATA_DIR / 'inegi_censo_2020' / Path(url).name
        print(download_file(url, output, max_size_mb=None))


### CONAGUA - disponibilidad de acuiferos

La pagina de CONAGUA publica una tabla HTML por estado. Esta celda descarga la tabla de Puebla, la convierte a `DataFrame` y filtra los acuiferos usados por el benchmark (`VALLE DE PUEBLA` y `ATLIXCO-IZUCAR DE MATAMOROS`).


In [ ]:
from html.parser import HTMLParser

class SimpleTableParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tables = []
        self._in_table = False
        self._in_row = False
        self._in_cell = False
        self._table = []
        self._row = []
        self._cell = []

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag == 'table':
            self._in_table = True
            self._table = []
        elif self._in_table and tag == 'tr':
            self._in_row = True
            self._row = []
        elif self._in_row and tag in {'td', 'th'}:
            self._in_cell = True
            self._cell = []

    def handle_data(self, data):
        if self._in_cell:
            self._cell.append(data)

    def handle_endtag(self, tag):
        tag = tag.lower()
        if self._in_cell and tag in {'td', 'th'}:
            self._row.append(' '.join(''.join(self._cell).split()))
            self._in_cell = False
        elif self._in_row and tag == 'tr':
            if self._row:
                self._table.append(self._row)
            self._in_row = False
        elif self._in_table and tag == 'table':
            self.tables.append(self._table)
            self._in_table = False

def read_conagua_acuiferos_estado(url: str) -> pd.DataFrame:
    response = http.get(url, timeout=60)
    response.raise_for_status()
    response.encoding = 'utf-8'

    parser = SimpleTableParser()
    parser.feed(response.text)
    tables = [table for table in parser.tables if len(table) > 1]
    if not tables:
        raise ValueError('No se encontro una tabla de acuiferos en la pagina de CONAGUA.')

    table = max(tables, key=len)
    df = pd.DataFrame(table[1:], columns=table[0])
    df = df.rename(columns={
        'CLAVE': 'clave',
        'ACUÍFERO': 'acuifero',
        'ACUĂ\x8dFERO': 'acuifero',
        'R': 'recarga_hm3_year',
        'DNC': 'dnc_hm3_year',
        'VEAS': 'veas_hm3_year',
        'DMA': 'dma_hm3_year',
        'DOCUMENTO': 'documento',
    })
    for column in ['recarga_hm3_year', 'dnc_hm3_year', 'veas_hm3_year', 'dma_hm3_year']:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    return df

conagua_puebla_url = 'https://sigagis.conagua.gob.mx/gas1/sections/Edos/puebla/puebla.html'
conagua_puebla_df = read_conagua_acuiferos_estado(conagua_puebla_url)

benchmark_acuifers = conagua_puebla_df[
    conagua_puebla_df['acuifero'].str.contains('VALLE DE PUEBLA|ATLIXCO', case=False, regex=True, na=False)
]

display(conagua_puebla_df)
display(benchmark_acuifers)
conagua_puebla_df.to_csv(DATA_DIR / 'conagua_acuiferos_puebla.csv', index=False)


### SIAP - cierre agricola

El sitio del SIAP puede tardar o no responder desde algunas redes. Esta celda intenta abrir la pagina con timeout corto; si falla, deja documentada la URL oficial y la carpeta local donde guardar manualmente el reporte exportado.


In [ ]:
siap_url = optional_data_sources['siap_cierre_agricola']
siap_dir = DATA_DIR / 'siap_cierre_agricola'
siap_dir.mkdir(parents=True, exist_ok=True)

try:
    response = http.get(siap_url, timeout=20)
    response.raise_for_status()
    (siap_dir / 'cierre_agricola_base.html').write_text(response.text, encoding='utf-8')
    siap_links = find_dataset_links(siap_url, extensions=('.zip', '.csv', '.xlsx', '.xls', '.json'))
    display(siap_links)
    if siap_links.empty:
        print('La pagina cargo, pero no expuso archivos directos. Genera el reporte desde la interfaz y guardalo en:', siap_dir)
except Exception as exc:
    siap_links = pd.DataFrame(columns=['text', 'url'])
    print('No se pudo conectar con SIAP:', type(exc).__name__, exc)
    print('Abre manualmente:', siap_url)
    print('Guarda los CSV/XLS exportados en:', siap_dir)

# Si obtienes una URL directa desde el navegador, pegala aqui y cambia DOWNLOAD_SIAP a True.
siap_report_url = ''
DOWNLOAD_SIAP = False
if DOWNLOAD_SIAP and siap_report_url:
    output = siap_dir / Path(siap_report_url.split('?')[0]).name
    print(download_file(siap_report_url, output, max_size_mb=None))


### REPDA - derechos de agua

REPDA es una aplicacion ASP.NET con `VIEWSTATE`. Esta celda no fuerza una busqueda automatica; guarda el HTML base, identifica campos de formulario y muestra el boton `Exportar`, para que puedas reproducir o completar la consulta desde navegador.


In [ ]:
import re
from urllib.parse import urljoin

repda_url = optional_data_sources['repda']
repda_dir = DATA_DIR / 'repda'
repda_dir.mkdir(parents=True, exist_ok=True)

repda_response = http.get(repda_url, timeout=60)
repda_response.raise_for_status()
repda_response.encoding = 'utf-8'
repda_html = repda_response.text
(repda_dir / 'consulta_repda_base.html').write_text(repda_html, encoding='utf-8')

repda_fields = []
for match in re.finditer(r'<(?:input|select|button)[^>]+>', repda_html, flags=re.I):
    tag = match.group(0)
    name = re.search(r'name=["\']([^"\']+)', tag, flags=re.I)
    element_id = re.search(r'id=["\']([^"\']+)', tag, flags=re.I)
    value = re.search(r'value=["\']([^"\']*)', tag, flags=re.I)
    if name or element_id:
        repda_fields.append({
            'name': name.group(1) if name else None,
            'id': element_id.group(1) if element_id else None,
            'value': value.group(1) if value else None,
            'tag': tag[:120],
        })

repda_fields_df = pd.DataFrame(repda_fields)
repda_export_fields = repda_fields_df[
    repda_fields_df.astype(str).apply(lambda row: row.str.contains('BtnExportar|CboEstado|CboMunicipio|CboUso|txtTitular|txtTitulo', case=False, regex=True).any(), axis=1)
]

display(repda_export_fields)
print('HTML base guardado en:', repda_dir / 'consulta_repda_base.html')
print('Para Puebla, el select CboEstado usa value=21. REPDA requiere postback ASP.NET para consultar/exportar resultados.')


### SMN - Monitor de Sequia en Mexico

La pagina del SMN incluye un XLSX directo de municipios con sequia y tambien PDFs/PNGs historicos del monitor. Esta celda extrae enlaces reales desde el HTML y deja la descarga controlada por banderas.


In [ ]:
import re
from urllib.parse import urljoin

smn_url = optional_data_sources['smn_monitor_sequia']
smn_dir = DATA_DIR / 'smn_monitor_sequia'
smn_dir.mkdir(parents=True, exist_ok=True)

smn_response = http.get(smn_url, timeout=60)
smn_response.raise_for_status()
smn_response.encoding = 'utf-8'
smn_html = smn_response.text
(smn_dir / 'monitor_sequia_base.html').write_text(smn_html, encoding='utf-8')

xlsx_links = sorted(set(urljoin(smn_url, href) for href in re.findall(r'href=["\']([^"\']+\.xlsx[^"\']*)', smn_html, flags=re.I)))
report_pdf_links = sorted(set(urljoin(smn_url, href) for href in re.findall(r"url=['\"]([^'\"]+\.pdf)['\"]", smn_html, flags=re.I)))
report_png_links = sorted(set(urljoin(smn_url, href) for href in re.findall(r"img=['\"]([^'\"]+\.png)['\"]", smn_html, flags=re.I)))

smn_links = pd.DataFrame(
    [{'kind': 'municipios_xlsx', 'url': url} for url in xlsx_links]
    + [{'kind': 'monitor_pdf', 'url': url} for url in report_pdf_links]
    + [{'kind': 'monitor_png', 'url': url} for url in report_png_links]
)
display(smn_links.head(30))

DOWNLOAD_SMN_MUNICIPIOS = False  # Cambia a True para bajar MunicipiosSequia.xlsx.
if DOWNLOAD_SMN_MUNICIPIOS and xlsx_links:
    url = xlsx_links[0]
    print(download_file(url, smn_dir / Path(url).name, max_size_mb=None))

DOWNLOAD_LATEST_SMN_REPORT = False  # Cambia a True para bajar el PDF mas reciente listado.
if DOWNLOAD_LATEST_SMN_REPORT and report_pdf_links:
    url = report_pdf_links[0]
    print(download_file(url, smn_dir / Path(url).name, max_size_mb=None))
